# EXIST 2026: sexism detection in memes, walkthrough

Reads the outputs of `src/` (the EXIST data itself is not redistributed), so this notebook runs without the dataset.

In [1]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.set_option("display.precision", 3)
json.load(open(ROOT / "results/eda_stats.json"))

{'memes': 3984,
 'by_language': {'en': 2005, 'es': 1979},
 'annotators_per_meme': 6,
 'unanimous_share': 0.2371987951807229,
 'tie_3_3_share': 0.15411646586345382,
 'median_words': 17.0,
 'memes_with_sensor_data': 1.0}

## 1. The data: short texts and a lot of human disagreement

![](../results/figures/eda_disagreement_and_length.png)

Only about 1 in 4 memes is labelled the same by all 6 annotators, and 15 % are a 3–3 tie. Sexism is partly subjective, which is why EXIST also scores *soft* predictions (section 5).

## 2. Twelve models, three tasks (5-fold cross-validation)

In [2]:
cv = pd.read_csv(ROOT / "results/cv_results_with_icm.csv", dtype={"task": str})
cv.pivot(index="model", columns="task", values="macro_f1").sort_values("2.1", ascending=False)

task,2.1,2.2,2.3
model,,,
ensemble_text_nb_lr_sensor,0.676,0.447,0.441
word_char_tfidf_logreg_balanced,0.675,0.452,0.456
word_char_tfidf_logreg,0.673,0.434,0.420
char_tfidf_logreg,0.672,0.420,0.411
word_char_tfidf_linearsvc,0.671,0.429,0.433
word_char_sensor_linearsvc,0.660,0.426,0.412
word_char_sensor_logreg_balanced,0.657,0.436,0.418
word_tfidf_complement_nb,0.654,0.428,0.431
word_char_sensor_logreg,0.653,0.440,0.407


![](../results/figures/models_by_task.png)

## 3. Before vs after: the model I submitted in May vs the best one now

In [3]:
pd.read_csv(ROOT / "results/before_vs_after.csv", dtype={"task": str})

,task,submitted_model,submitted_macro_f1,submitted_icm_norm,best_model,best_macro_f1,best_icm_norm
0,2.1,word_tfidf_complement_nb,0.654,0.478,ensemble_text_nb_lr_sensor,0.676,0.512
1,2.2,word_char_sensor_linearsvc,0.426,0.313,word_char_tfidf_logreg_balanced,0.452,0.334
2,2.3,char_tfidf_logreg,0.411,0.312,word_char_tfidf_logreg_balanced,0.456,0.295


## 4. Do the physiological signals help?

![](../results/figures/text_vs_sensors.png)

In [4]:
pd.read_csv(ROOT / "results/text_vs_sensors.csv")

,task,Text only,Text + sensors,Sensors only
0,2.1 · Is it sexist?,0.673,0.653,0.512
1,2.2 · Source intention,0.434,0.440,0.317
2,2.3 · Sexism category,0.420,0.407,0.266


## 5. Learning with disagreement (soft labels, Task 2.1)

Instead of YES/NO, predict the share of people who would call the meme sexist. Training on the vote shares themselves (each meme weighted as YES and NO) gives better-calibrated probabilities.

In [5]:
pd.read_csv(ROOT / "results/soft_labels_2_1.csv")

,model,mean_abs_error_vs_vote_share,cross_entropy,corr_with_vote_share
0,hard_trained,0.251,0.705,0.399
1,soft_trained,0.240,0.657,0.411


In [6]:
pd.DataFrame(json.load(open(ROOT / "results/pyevall_soft.json")))

,task,model,Information Contrast Model Soft,Normalized Information Contrast Model Soft,Cross Entropy
0,2.1-soft,hard_trained,-0.61,0.404,1.018
1,2.1-soft,soft_trained,-0.64,0.399,0.948


## 6. Errors and languages

![](../results/figures/confusion_matrices.png)

JUDGEMENTAL memes (criticising sexism) are mostly confused with DIRECT or NO: telling *reporting* from *endorsing* needs irony and context that a bag of words can't see.

In [7]:
pd.read_csv(ROOT / "results/per_language.csv").pivot(index="model", columns=["task", "lang"], values="macro_f1")

task                               2.1           2.2       
lang                                en     es     en     es
model                                                      
ensemble_text_nb_lr_sensor       0.695  0.646  0.462  0.420
word_char_sensor_logreg            NaN    NaN  0.454  0.420
word_char_tfidf_logreg           0.698  0.634    NaN    NaN
word_char_tfidf_logreg_balanced  0.690  0.653  0.478  0.412

## 7. Takeaways

* Well-tuned linear models on word + character n-grams are strong baselines for short, noisy OCR text.
* Class balancing is the cheapest big win for rare classes.
* Physiological reactions carry a weak signal about *intention*, not about *content*.
* Next: add the image (e.g. CLIP embeddings) and a multilingual transformer, evaluated under the same protocol.